### Creating CollectionBuilder compatible metadata improvements

- changing object_id to objectid
- adding 'filename' column as /article_id/file (removing objects/ in front)

In [1]:
import os
import pandas as pd


In [2]:

# --- CONFIG ---
INFILE = "complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = "cb_complete_metadata_images_tropes_reprints_transcripts.csv"  # safer new file
OVERWRITE = False  # set True to write back into INFILE
# ---------------

df = pd.read_csv(INFILE, dtype=str).fillna("")

# 1) Ensure CollectionBuilder-friendly objectid
if "objectid" not in df.columns:
    if "object_id" in df.columns:
        df = df.rename(columns={"object_id": "objectid"})
        print("Renamed column: object_id -> objectid")
    else:
        raise ValueError("Could not find 'objectid' or 'object_id' column in CSV.")

# 2) Add filename (relative to /objects/)
# Prefer image_object_location (your example); fall back to object_location if you use that.
path_col = None
for candidate in ["filename", "image_object_location", "object_location", "file", "filepath"]:
    if candidate in df.columns:
        path_col = candidate
        break

if "filename" not in df.columns:
    if path_col in ["image_object_location", "object_location"]:
        def to_filename(p: str) -> str:
            p = (p or "").strip()
            if not p:
                return ""
            # normalize slashes
            p = p.replace("\\", "/")
            # if it starts with objects/, strip it
            if p.startswith("objects/"):
                return p[len("objects/"):]
            # if it starts with /objects/, strip it
            if p.startswith("/objects/"):
                return p[len("/objects/"):]
            # otherwise assume it's already relative to objects/
            return p

        df["filename"] = df[path_col].apply(to_filename)
        print(f"Created 'filename' from '{path_col}' by stripping leading 'objects/'")
    elif path_col == "filename":
        # already exists, but this branch won't happen because we checked above
        pass
    else:
        raise ValueError(
            "Could not find a usable path column. Expected one of: "
            "'image_object_location' or 'object_location' (or already have 'filename')."
        )
else:
    print("Column 'filename' already exists—leaving as-is.")

# 3) Sanity checks: show a few examples
print("\nSample (objectid, filename):")
print(df[["objectid", "filename"]].head(10).to_string(index=False))

# 4) Write output
outfile = INFILE if OVERWRITE else OUTFILE
df.to_csv(outfile, index=False)
print(f"\nWrote: {outfile}")


Renamed column: object_id -> objectid
Created 'filename' from 'image_object_location' by stripping leading 'objects/'

Sample (objectid, filename):
                         objectid                                                       filename
AlbanyEveningJournal1853_Original AlbanyEveningJournal1853/AlbanyEveningJournal1853_DigSur_1.png
    AlbanyEveningJournal1853_img1 AlbanyEveningJournal1853/AlbanyEveningJournal1853_DigSur_1.png
    AlbanyEveningJournal1853_img2 AlbanyEveningJournal1853/AlbanyEveningJournal1853_DigSur_2.png
    AlbanyEveningJournal1853_img3 AlbanyEveningJournal1853/AlbanyEveningJournal1853_DigSur_3.png
    AlbanyEveningJournal1853_img4 AlbanyEveningJournal1853/AlbanyEveningJournal1853_DigSur_4.png
    AlbanyEveningJournal1853_img5 AlbanyEveningJournal1853/AlbanyEveningJournal1853_DigSur_5.png
    AlbanyEveningJournal1853_img6 AlbanyEveningJournal1853/AlbanyEveningJournal1853_DigSur_6.png
    AlbanyEveningJournal1853_img7 AlbanyEveningJournal1853/AlbanyEveningJour